# 02 · GGM Estimation — EBICglasso networks per context

## Model choice and specifications

We estimate regularized partial-correlation networks using the EBIC graphical lasso
(EBICglasso; Foygel & Drton, 2010) as implemented in `qgraph` / `bootnet`.

**Input:** Correlation matrix computed on complete-case data per context
(wave × trust stratum). Ordinal Likert items (1–7, 1–6) are treated as approximately
continuous; this is a deliberate simplification documented in the thesis.
The main specification uses Pearson correlations. A Spearman robustness check
is run to verify that key structural findings are not sensitive to this choice.

**Specifications run for every context:**

| Label | γ (EBIC) | Correlation | Thresholding | Bootstrap |
|-------|----------|-------------|--------------|-----------|
| `g05`         | 0.50 | Pearson  | No  | Yes (n = 1 000, nonparametric) |
| `g075`        | 0.75 | Pearson  | No  | Yes (n = 1 000, nonparametric) |
| `g05_thr`     | 0.50 | Pearson  | Yes | No  |
| `g05_spearman`| 0.50 | Spearman | No  | No  |

**Spearman specification (`g05_spearman`):** The full estimation pipeline is re-run
using Spearman rank correlations as input instead of Pearson correlations. This
serves as a robustness check for the ordinal-as-numeric GGM approximation: Spearman
correlations are rank-based and therefore robust to ceiling/floor effects, monotone
distortions, and departures from linearity. Bootstrap is skipped for this
specification because the primary comparison criterion is neighborhood composition
and focal-strength ordering, not edge-level stability.

**Thresholded specification (`g05_thr`):** After EBICglasso selects the network,
partial correlations that do not reach *p* < 0.05 under a Fisher *z*-test (analytic
approximation, *n* = complete-case sample size) are set to zero.  
This is implemented via `threshold = TRUE` in `bootnet::estimateNetwork` / `qgraph::EBICglasso`.  
It is used as a conservative robustness check only; bootstrap is skipped because the
thresholded network is not the primary estimand.

**EBICglasso sparsity warning:** When R prints  
*"Network with lowest lambda selected as best network: assumption of sparsity might  
be violated"*  
it means the EBIC criterion selected the weakest regularization in the default
search grid, suggesting the network may be denser than the sparsity prior implies.
Both γ values are therefore reported; conclusions are assessed against both.

**Outputs per context × specification:**  
- Fitted network object (`net_*.rds`)  
- Bootstrap object (`boot_edges_*.rds`) where applicable  
- Edge list CSV (`edges_*.csv`)  
- Adjacency matrix CSV (`adj_*.csv`)  
- Network plot PNG  
- Compact summary table (`networks_summary.csv`)


In [1]:
# ── Package availability check ────────────────────────────────────────────────
required_packages <- c("qgraph", "glasso", "bootnet")

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
  }
}

suppressPackageStartupMessages({
  library(qgraph)
  library(bootnet)
})

In [2]:
# ── Paths and index ───────────────────────────────────────────────────────────
INST_DIR   <- file.path("..", "data", "instances")
INDEX_PATH <- file.path(INST_DIR, "instances_index.csv")
OUT_DIR    <- file.path("..", "results", "networks")
PLOT_DIR   <- file.path(OUT_DIR, "plots")
dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)
dir.create(PLOT_DIR, recursive = TRUE, showWarnings = FALSE)

idx <- read.csv(INDEX_PATH, stringsAsFactors = FALSE)

# Verify index has the columns we rely on
required_idx_cols <- c("wave", "trust_stratum", "n_cc", "filename")
missing_idx_cols  <- setdiff(required_idx_cols, names(idx))
if (length(missing_idx_cols) > 0) {
  stop("instances_index.csv is missing columns: ",
       paste(missing_idx_cols, collapse = ", "),
       "\nRun 01_data_preprocessing first.")
}

cat("Index loaded:", nrow(idx), "instances\n")
print(idx)

Index loaded: 6 instances
  wave trust_stratum n_cc n_stratum                         filename
1   55          high  309       336 cosmo_nodes_wave55_trusthigh.csv
2   55           low  376       423  cosmo_nodes_wave55_trustlow.csv
3   56          high  351       360 cosmo_nodes_wave56_trusthigh.csv
4   56           low  383       431  cosmo_nodes_wave56_trustlow.csv
5   57          high  407       418 cosmo_nodes_wave57_trusthigh.csv
6   57           low  336       394  cosmo_nodes_wave57_trustlow.csv


In [3]:
# ── Node variables and estimation settings ────────────────────────────────────

NODE_VARS <- c(
  "SEVERITY",
  "AFF_FEAR", "AFF_WORRY", "WORRY_HEALTH_SYSTEM",
  "USE2_MASK", "USE2_SPACE150", "USE2_HANDWASH20", "USE2_AVOID"
)

# ── Specifications ────────────────────────────────────────────────────────────
# Each specification carries its own cor_method so that the estimation function
# can select the appropriate correlation input.
SPECS <- list(
  list(label = "g05",          gamma = 0.5,  threshold = FALSE, run_bootstrap = TRUE,  cor_method = "cor"),
  list(label = "g075",         gamma = 0.75, threshold = FALSE, run_bootstrap = TRUE,  cor_method = "cor"),
  list(label = "g05_thr",      gamma = 0.5,  threshold = TRUE,  run_bootstrap = FALSE, cor_method = "cor"),
  list(label = "g05_spearman", gamma = 0.5,  threshold = FALSE, run_bootstrap = FALSE, cor_method = "spearman")
)

# ── Bootstrap settings ────────────────────────────────────────────────────────
N_BOOTS    <- 1000L
BOOT_TYPE  <- "nonparametric"
BOOT_STATS <- c("edge")

# ── Reproducible seeding ──────────────────────────────────────────────────────
seed_net <- function(wave, stratum, spec_label) {
  s <- ifelse(tolower(stratum) %in% c("high", "h"), 2L, 1L)
  g <- switch(spec_label, "g075" = 7L, "g05_thr" = 6L, "g05_spearman" = 8L, 5L)
  as.integer(10000L + 100L * as.integer(wave) + 10L * s + g)
}

seed_boot <- function(wave, stratum, spec_label) {
  seed_net(wave, stratum, spec_label) + 50000L
}


In [4]:
# ── Core estimation function ──────────────────────────────────────────────────

estimate_and_save <- function(row_idx, gamma, spec_label,
                              threshold = FALSE, run_bootstrap = TRUE,
                              cor_method = "cor") {

  wave    <- idx$wave[row_idx]
  stratum <- idx$trust_stratum[row_idx]
  n_cc    <- idx$n_cc[row_idx]
  fname   <- idx$filename[row_idx]

  fpath <- file.path(INST_DIR, fname)
  if (!file.exists(fpath)) stop("Instance file not found: ", fpath)

  # ── Load and validate data ─────────────────────────────────────────────────
  dat <- read.csv(fpath, stringsAsFactors = FALSE)

  missing_nodes <- setdiff(NODE_VARS, names(dat))
  if (length(missing_nodes) > 0)
    stop("Missing node variables in ", fname, ": ", paste(missing_nodes, collapse = ", "))

  dat <- dat[, NODE_VARS, drop = FALSE]
  for (v in names(dat)) dat[[v]] <- as.numeric(dat[[v]])

  if (anyNA(dat))
    stop("Unexpected missing values in ", fname, " after node selection. ",
         "Ensure 01_data_preprocessing produced complete-case files.")

  if (nrow(dat) != n_cc)
    warning(sprintf("Row count in %s (%d) differs from index n_cc (%d).",
                    fname, nrow(dat), n_cc))

  tag <- sprintf("wave%d_trust%s_n%d_%s", wave, stratum, n_cc, spec_label)

  # ── Network estimation ─────────────────────────────────────────────────────
  if (cor_method == "spearman") {
    # Spearman path: compute rank-correlation matrix, then call
    # qgraph::EBICglasso directly. We bypass bootnet::estimateNetwork
    # because older versions do not support the corMatrix argument.
    cor_input <- cor(dat, method = "spearman")
    cor_label <- "spearman"

    set.seed(seed_net(wave, stratum, spec_label))
    W <- qgraph::EBICglasso(
      S     = cor_input,
      n     = nrow(dat),
      gamma = gamma,
      returnAllResults = FALSE
    )

    # Ensure row/col names match node variables
    if (is.null(rownames(W))) {
      rownames(W) <- colnames(cor_input)
      colnames(W) <- colnames(cor_input)
    }

    # Wrap in a list that mimics bootnet output structure for downstream code
    net <- list(graph = W, labels = NODE_VARS, nNodes = length(NODE_VARS))
    saveRDS(net, file.path(OUT_DIR, paste0("net_", tag, ".rds")))

  } else {
    # Pearson path: use bootnet::estimateNetwork as before
    cor_label <- "pearson"

    set.seed(seed_net(wave, stratum, spec_label))
    net <- bootnet::estimateNetwork(
      dat,
      default   = "EBICglasso",
      corMethod = cor_method,
      tuning    = gamma,
      threshold = threshold
    )
    W <- net$graph
    saveRDS(net, file.path(OUT_DIR, paste0("net_", tag, ".rds")))
  }

  # ── Bootstrap ─────────────────────────────────────────────────────────────
  if (run_bootstrap) {
    set.seed(seed_boot(wave, stratum, spec_label))
    boot_edges <- bootnet::bootnet(
      net,
      nBoots     = N_BOOTS,
      type       = BOOT_TYPE,
      statistics = BOOT_STATS
    )
    boot_rds  <- paste0("boot_edges_", tag, ".rds")
    saveRDS(boot_edges, file.path(OUT_DIR, boot_rds))
    boot_file_val <- boot_rds
  } else {
    boot_file_val <- NA_character_
  }

  # ── Edge list export ───────────────────────────────────────────────────────
  edge_df           <- as.data.frame(as.table(W), stringsAsFactors = FALSE)
  colnames(edge_df) <- c("from", "to", "weight")
  edge_df$from      <- as.character(edge_df$from)
  edge_df$to        <- as.character(edge_df$to)

  edge_df <- edge_df[edge_df$from < edge_df$to, ]
  edge_df <- edge_df[order(-abs(edge_df$weight)), ]

  write.csv(edge_df,
            file.path(OUT_DIR, paste0("edges_", tag, ".csv")),
            row.names = FALSE)

  # ── Adjacency matrix export ────────────────────────────────────────────────
  write.csv(W,
            file.path(OUT_DIR, paste0("adj_", tag, ".csv")),
            row.names = TRUE)

  # ── Summary row ──────────────────────────────────────────────────────────
  data.frame(
    wave                       = wave,
    trust_stratum              = stratum,
    n_cc                       = n_cc,
    spec                       = spec_label,
    gamma_ebic                 = gamma,
    threshold_applied          = threshold,
    cor_method                 = cor_label,
    n_edges_nonzero_undirected = sum(edge_df$weight != 0),
    n_boots                    = ifelse(run_bootstrap, N_BOOTS, NA_integer_),
    boot_type                  = ifelse(run_bootstrap, BOOT_TYPE, NA_character_),
    boot_file                  = boot_file_val,
    stringsAsFactors           = FALSE
  )
}

cat("estimate_and_save() defined.\n")


estimate_and_save() defined.


In [5]:
# ── Run all contexts × all specifications ─────────────────────────────────────
# Each call to estimate_and_save() handles one (context, spec) pair.

summaries <- list()
errors    <- list()

for (i in seq_len(nrow(idx))) {
  for (sp in SPECS) {
    label <- sprintf("wave=%d trust=%s spec=%s",
                     idx$wave[i], idx$trust_stratum[i], sp$label)
    cat("Estimating:", label, "\n")

    tryCatch({
      row <- estimate_and_save(
        row_idx       = i,
        gamma         = sp$gamma,
        spec_label    = sp$label,
        threshold     = sp$threshold,
        run_bootstrap = sp$run_bootstrap,
        cor_method    = sp$cor_method
      )
      summaries[[length(summaries) + 1]] <- row
    }, error = function(e) {
      msg <- paste0("[ERROR] ", label, ": ", conditionMessage(e))
      cat(msg, "\n")
      errors[[length(errors) + 1]] <<- msg
    })
  }
}

if (length(errors) > 0) {
  cat("\n--- Errors encountered ---\n")
  for (e in errors) cat(e, "\n")
  warning(length(errors), " estimation(s) failed — see output above.")
}

summaries_df <- do.call(rbind, summaries)
write.csv(summaries_df,
          file.path(OUT_DIR, "networks_summary.csv"),
          row.names = FALSE)

cat("\nEstimation complete.\n")
print(summaries_df)


Estimating: wave=55 trust=high spec=g05 


Estimating Network. Using package::function:
  - qgraph::EBICglasso for EBIC model selection
    - using glasso::glasso

Warning message in EBICglassoCore(S = S, n = n, gamma = gamma, penalize.diagonal = penalize.diagonal, :
"A dense regularized network was selected (lambda < 0.1 * lambda.max). Recent work indicates a possible drop in specificity. Interpret the presence of the smallest edges with care. Setting threshold = TRUE will enforce higher specificity, at the cost of sensitivity."
Note: bootnet will store only the following statistics:  edge

Bootstrapping...



  |=======                                                               |   9%

Note: Network with lowest lambda selected as best network: assumption of sparsity might be violated.



  |=================                                                     |  24%

...

  |======================================================================| 100%
Estimating: wave=57 trust=low spec=g05_thr 


Estimating Network. Using package::function:
  - qgraph::EBICglasso for EBIC model selection
    - using glasso::glasso

Note: Network with lowest lambda selected as best network: assumption of sparsity might be violated.



Estimating: wave=57 trust=low spec=g05_spearman 


Warning message in EBICglassoCore(S = S, n = n, gamma = gamma, penalize.diagonal = penalize.diagonal, :
"A dense regularized network was selected (lambda < 0.1 * lambda.max). Recent work indicates a possible drop in specificity. Interpret the presence of the smallest edges with care. Setting threshold = TRUE will enforce higher specificity, at the cost of sensitivity."



Estimation complete.
   wave trust_stratum n_cc         spec gamma_ebic threshold_applied cor_method
1    55          high  309          g05       0.50             FALSE    pearson
2    55          high  309         g075       0.75             FALSE    pearson
3    55          high  309      g05_thr       0.50              TRUE    pearson
4    55          high  309 g05_spearman       0.50             FALSE   spearman
5    55           low  376          g05       0.50             FALSE    pearson
6    55           low  376         g075       0.75             FALSE    pearson
7    55           low  376      g05_thr       0.50              TRUE    pearson
8    55           low  376 g05_spearman       0.50             FALSE   spearman
9    56          high  351          g05       0.50             FALSE    pearson
10   56          high  351         g075       0.75             FALSE    pearson
11   56          high  351      g05_thr       0.50              TRUE    pearson
12   56          h

## Structural diagnostic: behavioral node embeddedness

To verify that behavioral nodes (`USE2_*`) are not isolated under regularization,
we compute **degree** (number of non-zero edges) and **strength** (sum of absolute
edge weights) for each behavioral node across all specifications and contexts.

A degree of 0 in the main specification (`g05`) indicates a node that is structurally
disconnected from the belief–affect subgraph and cannot contribute to bridge analysis
or embeddedness metrics for the focal belief.  Such nodes would need to be flagged in
the thesis discussion.

In [6]:
# ── Behavioral node degree and strength across all contexts × specs ───────────

BEHAV_NODES <- c("USE2_MASK", "USE2_SPACE150", "USE2_HANDWASH20", "USE2_AVOID")

get_node_stats <- function(W) {
  diag(W) <- 0
  data.frame(
    node     = rownames(W),
    degree   = as.integer(rowSums(W != 0)),
    strength = as.numeric(rowSums(abs(W))),
    stringsAsFactors = FALSE
  )
}

beh_stats_rows <- list()

for (i in seq_len(nrow(idx))) {
  wave    <- idx$wave[i]
  stratum <- idx$trust_stratum[i]
  n_cc    <- idx$n_cc[i]

  for (sp in SPECS) {
    tag      <- sprintf("wave%d_trust%s_n%d_%s", wave, stratum, n_cc, sp$label)
    obj_path <- file.path(OUT_DIR, paste0("net_", tag, ".rds"))
    if (!file.exists(obj_path)) {
      cat("Skipping (file not found):", obj_path, "\n")
      next
    }
    obj   <- readRDS(obj_path)
    W     <- if (is.list(obj) && !is.null(obj$graph)) obj$graph else obj
    stats <- get_node_stats(W)
    stats <- stats[stats$node %in% BEHAV_NODES, ]
    stats$wave          <- wave
    stats$trust_stratum <- stratum
    stats$n_cc          <- n_cc
    stats$spec          <- sp$label
    stats$gamma         <- sp$gamma
    beh_stats_rows[[length(beh_stats_rows) + 1]] <- stats
  }
}

beh_stats <- do.call(rbind, beh_stats_rows)
beh_stats <- beh_stats[order(beh_stats$spec, beh_stats$wave,
                              beh_stats$trust_stratum, beh_stats$node), ]
rownames(beh_stats) <- NULL

cat("\nBehavioral node statistics (degree / strength):\n")
print(beh_stats)

isolated <- subset(beh_stats, degree == 0)
if (nrow(isolated) > 0) {
  cat("\n⚠ Isolated behavioral nodes (degree = 0):\n")
  print(isolated[, c("wave","trust_stratum","spec","node","degree","strength")])
} else {
  cat("\n✓ No isolated behavioral nodes detected in any context × spec.\n")
}


Behavioral node statistics (degree / strength):
              node degree  strength wave trust_stratum n_cc         spec gamma
1       USE2_AVOID      7 0.7157924   55          high  309          g05  0.50
2  USE2_HANDWASH20      7 0.6438941   55          high  309          g05  0.50
3        USE2_MASK      6 0.6262256   55          high  309          g05  0.50
4    USE2_SPACE150      5 0.8069789   55          high  309          g05  0.50
5       USE2_AVOID      7 0.7738490   55           low  376          g05  0.50
6  USE2_HANDWASH20      5 0.6462332   55           low  376          g05  0.50
7        USE2_MASK      6 0.8040627   55           low  376          g05  0.50
8    USE2_SPACE150      5 0.9187711   55           low  376          g05  0.50
9       USE2_AVOID      7 0.7674004   56          high  351          g05  0.50
10 USE2_HANDWASH20      5 0.6883496   56          high  351          g05  0.50
11       USE2_MASK      4 0.5150406   56          high  351          g05  0.50
12 

## Network visualization

Spring-layout plots are generated for every context × specification and saved as
PNG files.  These are descriptive visualizations; edge widths represent absolute
partial-correlation weights, green = positive, red = negative.

In [7]:
# ── Network plots ─────────────────────────────────────────────────────────────

make_plot <- function(W, title, out_file) {
  diag(W) <- 0
  png(out_file, width = 1600, height = 1200, res = 200)
  on.exit(dev.off())   # guarantees device is closed even if qgraph errors
  set.seed(2024)       # fixed seed for reproducible spring layouts
  qgraph(
    W,
    layout    = "spring",
    labels    = colnames(W),
    title     = title,
    vsize     = 6,
    label.cex = 0.9
  )
}

for (i in seq_len(nrow(idx))) {
  wave    <- idx$wave[i]
  stratum <- idx$trust_stratum[i]
  n_cc    <- idx$n_cc[i]

  for (sp in SPECS) {
    tag      <- sprintf("wave%d_trust%s_n%d_%s", wave, stratum, n_cc, sp$label)
    obj_path <- file.path(OUT_DIR, paste0("net_", tag, ".rds"))
    if (!file.exists(obj_path)) next

    obj <- readRDS(obj_path)
    W   <- if (is.list(obj) && !is.null(obj$graph)) obj$graph else obj

    thr_label <- if (sp$threshold) " [thresholded]" else ""
    make_plot(
      W,
      sprintf("EBICglasso (γ = %.2f%s) — wave %d, %s trust (n = %d)",
              sp$gamma, thr_label, wave, stratum, n_cc),
      file.path(PLOT_DIR, paste0("plot_net_", tag, ".png"))
    )
  }
}

cat("Plots saved to:", PLOT_DIR, "\n")

Plots saved to: ../results/networks/plots 


## Spearman vs. Pearson comparison

The following cell generates a comparison table of focal strength, global
strength, and top-ranked SEVERITY neighbor under Pearson vs. Spearman
correlation inputs. This table maps directly to **Table V.X** in the thesis
(`tab:spearman_comparison`) and provides the values reported in the robustness section of the thesis
(`tab:spearman_comparison`).


In [8]:
# ── Spearman vs. Pearson comparison table ─────────────────────────────────────

comparison_rows <- list()

for (i in seq_len(nrow(idx))) {
  wave    <- idx$wave[i]
  stratum <- idx$trust_stratum[i]
  n_cc    <- idx$n_cc[i]

  tag_pear  <- sprintf("wave%d_trust%s_n%d_g05", wave, stratum, n_cc)
  tag_spear <- sprintf("wave%d_trust%s_n%d_g05_spearman", wave, stratum, n_cc)

  net_pear_path  <- file.path(OUT_DIR, paste0("net_", tag_pear, ".rds"))
  net_spear_path <- file.path(OUT_DIR, paste0("net_", tag_spear, ".rds"))

  if (!file.exists(net_pear_path)) {
    cat("Skipping (Pearson not found):", tag_pear, "\n"); next
  }
  if (!file.exists(net_spear_path)) {
    cat("Skipping (Spearman not found):", tag_spear, "\n"); next
  }

  # Load adjacency matrices — handle both bootnet objects and raw lists
  obj_pear  <- readRDS(net_pear_path)
  obj_spear <- readRDS(net_spear_path)
  W_pear  <- if (!is.null(obj_pear$graph))  obj_pear$graph  else obj_pear
  W_spear <- if (!is.null(obj_spear$graph)) obj_spear$graph else obj_spear

  # Helper: compute focal strength, GS, top neighbor
  focal_stats <- function(W) {
    diag(W) <- 0
    focal_row <- abs(W["SEVERITY", ])
    focal_row <- focal_row[names(focal_row) != "SEVERITY"]
    S_focal <- sum(focal_row)
    GS      <- sum(abs(W[upper.tri(W)]))
    top_nb  <- if (S_focal > 0) names(which.max(focal_row)) else "none"
    list(S_focal = round(S_focal, 3), GS = round(GS, 3), top_neighbor = top_nb)
  }

  fp <- focal_stats(W_pear)
  fs <- focal_stats(W_spear)

  comparison_rows[[length(comparison_rows) + 1]] <- data.frame(
    wave             = wave,
    trust_stratum    = stratum,
    n_cc             = n_cc,
    S_focal_pearson  = fp$S_focal,
    S_focal_spearman = fs$S_focal,
    GS_pearson       = fp$GS,
    GS_spearman      = fs$GS,
    top_nb_pearson   = fp$top_neighbor,
    top_nb_spearman  = fs$top_neighbor,
    nb_match         = fp$top_neighbor == fs$top_neighbor,
    stringsAsFactors = FALSE
  )
}

if (length(comparison_rows) == 0) {
  cat("\nNo Spearman networks found. Run the full notebook top-to-bottom.\n")
} else {
  comp_df <- do.call(rbind, comparison_rows)
  comp_df <- comp_df[order(comp_df$wave, comp_df$trust_stratum), ]
  rownames(comp_df) <- NULL

  write.csv(comp_df,
            file.path(OUT_DIR, "spearman_vs_pearson_comparison.csv"),
            row.names = FALSE)

  cat("\n", paste(rep("=", 67), collapse=""), "\n")
  cat("Spearman vs. Pearson comparison (thesis Table tab:spearman_comparison)\n")
  cat(paste(rep("=", 67), collapse=""), "\n\n")
  print(comp_df)

  n_match <- sum(comp_df$nb_match)
  n_total <- nrow(comp_df)
  cat(sprintf("\nTop-neighbor match: %d / %d contexts\n", n_match, n_total))

  rs <- cor(comp_df$S_focal_pearson, comp_df$S_focal_spearman, method = "spearman")
  cat(sprintf("Spearman rank correlation of S(SEVERITY): rs = %.3f\n", rs))

  cat("\nSaved: spearman_vs_pearson_comparison.csv\n")
}



Spearman vs. Pearson comparison (thesis Table tab:spearman_comparison)

  wave trust_stratum n_cc S_focal_pearson S_focal_spearman GS_pearson
1   55          high  309           0.753            0.591      2.948
2   55           low  376           0.587            0.571      3.177
3   56          high  351           0.503            0.531      2.656
4   56           low  383           0.717            0.683      3.049
5   57          high  407           0.364            0.315      2.303
6   57           low  336           0.554            0.506      3.265
  GS_spearman top_nb_pearson top_nb_spearman nb_match
1       2.687       AFF_FEAR        AFF_FEAR     TRUE
2       3.333       AFF_FEAR        AFF_FEAR     TRUE
3       2.810       AFF_FEAR        AFF_FEAR     TRUE
4       3.040      AFF_WORRY        AFF_FEAR    FALSE
5       1.999       AFF_FEAR        AFF_FEAR     TRUE
6       3.131      AFF_WORRY       AFF_WORRY     TRUE

Top-neighbor match: 5 / 6 contexts
Spearman rank correlati

## Session info

`sessionInfo()` is exported to a text file so that exact package versions can be
recorded alongside results and committed to the repository.

In [9]:
# ── Export R session information ──────────────────────────────────────────────
session_path <- file.path(OUT_DIR, "r_session_info.txt")
sink(session_path)
cat("R session info — 02_ggm_estimation\n")
cat("Generated:", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"), "\n\n")
print(sessionInfo())
sink()
cat("Saved session info to:", session_path, "\n")

Saved session info to: ../results/networks/r_session_info.txt 


In [10]:
# ── CS-coefficient (case-dropping stability for strength centrality) ──────────
# Added as a separate cell after the main estimation loop.
# Uses the same seed_boot() function and pipeline settings already defined above.

suppressPackageStartupMessages(library(bootnet))

cs_results <- list()

for (i in seq_len(nrow(idx))) {

  wave    <- idx$wave[i]
  stratum <- idx$trust_stratum[i]
  n_cc    <- idx$n_cc[i]
  fname   <- idx$filename[i]

  dat <- read.csv(file.path(INST_DIR, fname), stringsAsFactors = FALSE)
  dat <- dat[, NODE_VARS, drop = FALSE]
  for (v in names(dat)) dat[[v]] <- as.numeric(dat[[v]])

  cat(sprintf("CS bootstrap: wave=%d trust=%s n=%d\n", wave, stratum, n_cc))

  # Use seed_boot() with spec_label "g05" — consistent with main estimation
  set.seed(seed_boot(wave, stratum, "g05") + 99999L)  # +offset avoids collision with edge bootstrap

  boot_case <- bootnet::bootnet(
    dat,
    nBoots     = 1000L,
    type       = "case",
    statistics = "strength",
    default    = "EBICglasso",
    corMethod  = "cor",  # Pearson — CS is always computed on the main specification
    tuning     = 0.5
  )

  cs_val <- corStability(boot_case, statistics = "strength", verbose = FALSE)

  cs_results[[length(cs_results) + 1]] <- data.frame(
    wave          = wave,
    trust_stratum = stratum,
    n_cc          = n_cc,
    CS_strength   = round(cs_val, 3),
    stringsAsFactors = FALSE
  )

  cat(sprintf("  CS-coefficient: %.3f\n", cs_val))
}

cs_table <- do.call(rbind, cs_results)
cs_table <- cs_table[order(cs_table$wave, cs_table$trust_stratum), ]

write.csv(cs_table,
          file.path(OUT_DIR, "cs_coefficients.csv"),
          row.names = FALSE)

print(cs_table)
cat("Saved: cs_coefficients.csv\n")

CS bootstrap: wave=55 trust=high n=309


Note: bootnet will store only the following statistics:  strength

Estimating sample network...

Estimating Network. Using package::function:
  - qgraph::EBICglasso for EBIC model selection
    - using glasso::glasso

Warning message in EBICglassoCore(S = S, n = n, gamma = gamma, penalize.diagonal = penalize.diagonal, :
"A dense regularized network was selected (lambda < 0.1 * lambda.max). Recent work indicates a possible drop in specificity. Interpret the presence of the smallest edges with care. Setting threshold = TRUE will enforce higher specificity, at the cost of sensitivity."
Bootstrapping...



  |================================                                      |  45%

Note: Network with lowest lambda selected as best network: assumption of sparsity might be violated.



  |========================================                              |  57%

...

  |============================================================          |  86%

Note: Network with lowest lambda selected as best network: assumption of sparsity might be violated.



  |======================================================================| 100%


Computing statistics...



  |======================================================================| 100%
  CS-coefficient: 0.595
          wave trust_stratum n_cc CS_strength
strength    55          high  309       0.440
strength1   55           low  376       0.516
strength2   56          high  351       0.516
strength3   56           low  383       0.671
strength4   57          high  407       0.595
strength5   57           low  336       0.595
Saved: cs_coefficients.csv
